<a href="https://colab.research.google.com/github/Shajs23224224/FFAIAssistant/blob/main/Copy_of_FFAI_Colab_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎮 Free Fire AI - Colab + Drive + Flask + Ngrok v3.2

**Mejoras v3.2:**
- ✅ WebSocket estable con eventlet (elimina errores `write() before start_response`)
- ✅ Heartbeat automático para detectar desconexiones
- ✅ Token Ngrok seguro (input oculto o variable de entorno)
- ✅ Ping/pong cada 15s para mantener conexión viva

**Arquitectura:**
- 🧠 Colab = Motor IA (GPU Tesla T4)
- 💾 Drive = Persistencia modelos/logs
- 🌐 Flask API = Interfaz REST/WebSocket (namespace `/ffai`)
- 🔗 Ngrok = URL pública
- 📱 Cliente = APK Android

## Instrucciones:
1. **Runtime → Change runtime type → GPU**
2. **Runtime → Run all** (Ctrl+F9)
3. Ingresa token Ngrok cuando se solicite
4. Copia la URL en ServerConfig.kt del APK
5. ¡Juega!

## Celda 1: Montar Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive montado en /content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive montado en /content/drive


## Celda 2: Instalar Dependencias

In [2]:
!pip install -q flask flask-socketio flask-cors pyngrok torch torchvision pillow numpy opencv-python-headless eventlet
print("✅ Dependencias instaladas (incluyendo eventlet para WebSocket estable)")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
onnx2tf 2.4.0 requires numpy==1.26.4, but you have numpy 2.4.4 which is incompatible.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.25.5 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.4 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.4 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.5 which is incompatible.
✅ Dependencias instaladas (incluyendo eventlet para WebSocket estable)


## Celda 3: Crear Módulos del Sistema

In [3]:
# Crear estructura de directorios
import os
os.makedirs('/content/ffai_core', exist_ok=True)

# Guardar engine.py
engine_code = '''"""Motor de IA"""

import torch
import torch.nn as nn
import numpy as np
import cv2
from PIL import Image
import io
import base64
import time
from typing import Dict, Tuple, List


class TacticalNN(nn.Module):
    def __init__(self, hidden=256, actions=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(128, hidden), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden, 128), nn.ReLU(),
            nn.Linear(128, actions)
        )

    def forward(self, x):
        return torch.softmax(self.net(x), dim=-1)


class FFAIEngine:
    ACTIONS = {
        0: ("HOLD", 960, 700, 100),
        1: ("SHOOT", 1150, 750, 100),
        2: ("AIM", 1150, 600, 200),
        3: ("FORWARD", 960, 500, 300),
        4: ("BACK", 960, 900, 300),
        5: ("LEFT", 700, 700, 300),
        6: ("RIGHT", 1220, 700, 300),
        7: ("RELOAD", 1200, 900, 1500),
        8: ("HEAL", 150, 800, 500),
        9: ("JUMP", 960, 700, 100),
    }

    def __init__(self, device="cpu"):
        self.device = torch.device(device)
        self.model = TacticalNN().to(self.device)
        self.model.eval()
        self.frame_count = 0
        self.session_start = time.time()
        print(f"🧠 Engine listo en {device}")

    def decode_image(self, b64):
        try:
            img_bytes = base64.b64decode(b64)
            img = Image.open(io.BytesIO(img_bytes))
            return np.array(img)
        except:
            return np.zeros((135, 240, 3), dtype=np.uint8)

    def analyze_frame(self, img):
        h, w = img.shape[:2]
        health_zone = img[int(h*0.88):int(h*0.96), int(w*0.02):int(w*0.22)]
        ammo_zone = img[int(h*0.88):int(h*0.96), int(w*0.78):int(w*0.98)]
        enemy_zone = img[int(h*0.15):int(h*0.55), int(w*0.35):int(w*0.65)]

        hsv = cv2.cvtColor(health_zone, cv2.COLOR_RGB2HSV)
        green = cv2.inRange(hsv, np.array([35,40,40]), np.array([85,255,255]))
        health = min(np.sum(green>0)/green.size*3, 1.0)

        gray = cv2.cvtColor(ammo_zone, cv2.COLOR_RGB2GRAY)
        ammo = min(np.sum(gray>200)/gray.size*3, 1.0)

        hsv = cv2.cvtColor(enemy_zone, cv2.COLOR_RGB2HSV)
        r1 = cv2.inRange(hsv, np.array([0,50,50]), np.array([10,255,255]))
        r2 = cv2.inRange(hsv, np.array([160,50,50]), np.array([180,255,255]))
        enemy = (np.sum(cv2.bitwise_or(r1,r2)>0) / hsv.size) > 0.08

        return {"health": float(health), "ammo": float(ammo), "enemy": bool(enemy)}

    def process(self, image_b64, client_state=None):
        self.frame_count += 1
        img = self.decode_image(image_b64)
        state = self.analyze_frame(img)

        # Heurísticas
        if state["health"] < 0.25:
            return {"action": "HEAL", "x": 150, "y": 800, "duration": 500, "state": state}
        if state["enemy"] and state["ammo"] < 0.1:
            return {"action": "RELOAD", "x": 1200, "y": 900, "duration": 1500, "state": state}
        if state["enemy"]:
            return {"action": "SHOOT", "x": 1150, "y": 750, "duration": 100, "state": state}

        # IA
        vec = torch.FloatTensor([state["health"], state["ammo"], float(state["enemy"]), 1.0, 0.5]).to(self.device)
        with torch.no_grad():
            probs = self.model(torch.cat([vec, torch.zeros(123).to(self.device)]))
            action_idx = torch.multinomial(probs, 1).item()

        name, x, y, dur = self.ACTIONS.get(action_idx, ("HOLD", 960, 700, 100))
        return {"action": name, "x": x, "y": y, "duration": dur, "state": state}
'''

with open('/content/ffai_core/engine.py', 'w') as f:
    f.write(engine_code)

print("✅ engine.py creado")

✅ engine.py creado


## Celda 4: Configurar GPU

In [4]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"🚀 GPU Tesla T4 activa")
    print(f"🔥 Memoria: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("⚠️ Usando CPU (más lento)")
    print("💡 Runtime → Change runtime type → GPU")

/usr/local/lib/python3.12/dist-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: module 'numpy._globals' has no attribute '_signature_descriptor' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


🚀 GPU Tesla T4 activa
🔥 Memoria: 15.6 GB


## Celda 5: Inicializar Motor IA

In [6]:
# 1) Reinstala dependencias para corregir problemas de entorno con numpy/cv2
!pip -q install --upgrade --force-reinstall numpy opencv-python-headless

# IMPORTANTE: Después de ejecutar esta celda, debes REINICIAR EL ENTORNO (Runtime -> Restart runtime)
# y luego volver a ejecutar todas las celdas.

import os
from ffai_core.engine import FFAIEngine

# Crear motor
engine = FFAIEngine(device=device)

# Intentar cargar modelo desde Drive si existe
model_path = "/content/drive/MyDrive/FFAI/models/ff_model.pt"
if os.path.exists(model_path):
    print(f"📂 Cargando modelo desde Drive...")
    # engine.load_model(model_path)  # Implementar si hay modelos guardados
else:
    print(f"💾 El modelo se guardará en: {model_path}")
    os.makedirs(os.path.dirname(model_path), exist_ok=True)

print("✅ Motor IA listo")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 12.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
onnx2tf 2.4.0 requires numpy==1.26.4, but you have numpy 2.4.4 which is incompatible.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.25.5 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.4 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.4 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.5 which is incompatible.


ImportError: cannot load module more than once per process

ImportError: numpy._core.multiarray failed to import

## Celda 6: Crear API Flask

In [ ]:
from flask import Flask, request, jsonify, send_file
from flask_socketio import SocketIO, emit, join_room, leave_room
from flask_cors import CORS
import json
import time
import os

app = Flask(__name__)
CORS(app)
socketio = SocketIO(app, cors_allowed_origins="*", async_mode="eventlet", ping_timeout=30, ping_interval=15)

# Configuration
MODELS_DIR = "/content/drive/MyDrive/FFAI/models"
os.makedirs(MODELS_DIR, exist_ok=True)

model_versions = {
    "perception": {"v1": f"{MODELS_DIR}/perception.tflite"},
    "policy": {"v1": f"{MODELS_DIR}/policy.tflite"},
    "economy": {"v1": f"{MODELS_DIR}/economy.tflite"},
}

# Stats
stats = {
    "started": time.time(),
    "frames_processed": 0,
    "clients_connected": 0,
    "room_count": 0,
    "last_heartbeat": {}
}

# ============================================
# HTTP ROUTES
# ============================================

@app.route("/")
def root():
    return jsonify({
        "status": "Online",
        "service": "FFAI API v3.2",
        "stats": {"frames": stats["frames_processed"], "clients": stats["clients_connected"]}
    })

@app.route("/models/list", methods=["GET"])
def list_models():
    available = {}
    for name, versions in model_versions.items():
        available[name] = {
            "versions": list(versions.keys()),
            "sizes_kb": {v: os.path.getsize(p)/1024 if os.path.exists(p) else 0 for v, p in versions.items()}
        }
    return jsonify(available)

@app.route("/models/<model_name>/<version>", methods=["GET"])
def download_model(model_name, version):
    path = model_versions.get(model_name, {}).get(version)
    if not path or not os.path.exists(path):
        return jsonify({"error": "Model file not found"}), 404
    return send_file(path, as_attachment=True)

# ============================================
# SOCKETIO EVENTS - NAMESPACE /ffai
# ============================================

@socketio.on("connect", namespace="/ffai")
def handle_ffai_connect():
    stats["clients_connected"] += 1
    join_room(f"client_{request.sid}")
    print(f"Cliente conectado: {request.sid}")

@socketio.on("frame", namespace="/ffai")
def handle_ffai_frame(data):
    try:
        if isinstance(data, str): data = json.loads(data)
        result = engine.process(data.get("imageBase64", ""), data.get("clientState", {}))
        stats["frames_processed"] += 1
        emit("action", {"action": result["action"], "coordinates": {"x": result["x"], "y": result["y"]}}, room=f"client_{request.sid}")
    except Exception as e:
        print(f"Error: {e}")

print("API Flask-SocketIO configurada con rutas de distribucion")

## Celda 7: Conectar Ngrok (URL Pública)

In [ ]:
from pyngrok import ngrok
import nest_asyncio
import os
from getpass import getpass

# Patch para asyncio
nest_asyncio.apply()

# Establecer el token de Ngrok proporcionado por el usuario
NGROK_TOKEN = "3CU4gdr4ZZBNCaiS2SS0M65EZZH_55e54r4qkAkq8aEu2MQE8"

ngrok.set_auth_token(NGROK_TOKEN)

# Crear túnel
public_url = ngrok.connect(5000, "http")
ngrok_domain = public_url.public_url.replace("https://", "").replace("http://", "")

print("\n" + "="*60)
print("🌐 URL PÚBLICA DEL SERVIDOR:")
print(f"   {public_url}")
print("="*60)
print(f"\n📋 Configura tu APK:")
print(f"   URL: {ngrok_domain}")
print(f"   Namespace: /ffai")
print(f"\n⚡ El servidor usará eventlet para estabilidad WebSocket")
print(f"✅ Listo para conexiones!")

## Celda 8: 🚀 INICIAR SERVIDOR

In [ ]:
import eventlet

print("🚀 Iniciando servidor FFAI con eventlet...")
print("   WebSocket estable con heartbeat automático")
print("   Presiona STOP para detener\n")

# Monkey patch para que eventlet maneje todo
# eventlet.monkey_patch()  # Descomentar si hay problemas de compatibilidad

# Iniciar con eventlet (más estable que threading)
# Comentar esta línea para que el servidor inicie después de definir todos los endpoints (celda 19)
# socketio.run(app, host="0.0.0.0", port=5000, debug=False)

## Celda 9: 🎓 Training Pipeline - Model Training & Distillation

Celdas adicionales para entrenamiento, destilación y exportación a TFLite.
Ejecutar cuando tengas suficientes experiencias recolectadas.

In [ ]:
# Celda 10: Training - Perception CNN (Teacher Model)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

class PerceptionCNN(nn.Module):
    """Teacher CNN - Modelo grande para detección de enemigos/HUD"""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 12 * 20, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128),
        )
        # Outputs: heatmap 20x12 + HUD 4 valores
        self.heatmap_head = nn.Linear(128, 240)  # 20x12 grid
        self.hud_head = nn.Linear(128, 4)         # health, ammo, coins, armor

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        heatmap = torch.sigmoid(self.heatmap_head(x))
        hud = torch.sigmoid(self.hud_head(x))
        return heatmap, hud

print("✅ PerceptionCNN (Teacher) definida")
print("   Input: 160x96x3 RGB")
print("   Output: heatmap 20x12 + HUD 4 valores")

# Instanciar modelo teacher
teacher_model = PerceptionCNN().to(device)
teacher_params = sum(p.numel() for p in teacher_model.parameters())
print(f"📊 Parámetros teacher: {teacher_params:,} (~{teacher_params*4/1024/1024:.1f}MB fp32)")

In [ ]:
# Celda 11: Knowledge Distillation - Student Model

class StudentPerceptionCNN(nn.Module):
    """Student CNN - Modelo pequeño para móvil (distilled from teacher)"""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 24 * 40, 64), nn.ReLU(),
            nn.Linear(64, 64),
        )
        self.heatmap_head = nn.Linear(64, 240)
        self.hud_head = nn.Linear(64, 4)

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        heatmap = torch.sigmoid(self.heatmap_head(x))
        hud = torch.sigmoid(self.hud_head(x))
        return heatmap, hud

student_model = StudentPerceptionCNN().to(device)
student_params = sum(p.numel() for p in student_model.parameters())
print(f"✅ StudentPerceptionCNN definida")
print(f"   Parámetros student: {student_params:,} (~{student_params*4/1024/1024:.1f}MB fp32)")
print(f"   Compresión: {teacher_params/student_params:.1f}x smaller than teacher")

In [ ]:
# Celda 12: Policy Model - Decision MLP

class PolicyMLP(nn.Module):
    """MLP for tactical decisions (32 inputs → 15 actions)"""
    def __init__(self, input_dim=32, hidden_dim=64, actions=15):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim//2), nn.ReLU(),
            nn.Linear(hidden_dim//2, actions + 4),  # 15 actions + x,y,duration,confidence
        )

    def forward(self, x):
        return self.net(x)

policy_model = PolicyMLP().to(device)
policy_params = sum(p.numel() for p in policy_model.parameters())
print(f"✅ PolicyMLP definida")
print(f"   Input: {32} features (estado del juego)")
print(f"   Output: {15} acciones + 4 parámetros")
print(f"   Parámetros: {policy_params:,} (~{policy_params*4/1024/1024:.1f}MB fp32)")

In [ ]:
# Celda 13 ULTRA ROBUSTA AUTO-DETECT (corregida)

import os
import sys
import shutil
import tempfile
import subprocess
import torch
import tensorflow as tf

# Instala dependencias si hace falta
!pip install -q onnx onnxscript onnx2tf

# ------------------------
# Helpers
# ------------------------

def find_saved_model(root):
    for d, _, files in os.walk(root):
        if "saved_model.pb" in files:
            return d
    return None

def find_all_tflites(root):
    found = []
    for d, _, files in os.walk(root):
        for f in files:
            if f.endswith(".tflite"):
                found.append(os.path.join(d, f))
    return found

def score_tflite(path):
    name = path.lower()
    score = 0

    if "int8" in name:
        score += 100
    if "float16" in name:
        score += 60
    if "float32" in name:
        score += 30

    size = os.path.getsize(path)
    score += max(0, 50_000_000 - size) / 1e6
    return score

def choose_best_tflite(paths):
    if not paths:
        return None
    scored = [(score_tflite(p), p) for p in paths]
    scored.sort(reverse=True)
    return scored[0][1]

def validate_tflite(path):
    try:
        interpreter = tf.lite.Interpreter(model_path=path)
        interpreter.allocate_tensors()
        return True
    except Exception as e:
        print("⚠️ Validación TFLite falló:", e)
        return False

# ------------------------
# Export principal
# ------------------------

def export_to_tflite_auto(model, input_shape, output_path, representative_dataset=None):
    model.eval()

    first_param = next(model.parameters(), None)
    # CORRECCIÓN: Usar 'is not None' para evitar ambigüedad con el tensor
    device = first_param.device if first_param is not None else torch.device("cpu")

    fd, temp_onnx = tempfile.mkstemp(suffix=".onnx")
    os.close(fd)
    temp_tf_dir = tempfile.mkdtemp(prefix="onnx2tf_")

    try:
        x = torch.randn(1, *input_shape, device=device)

        if torch.isnan(x).any():
            raise RuntimeError("Dummy input contiene NaN")

        # --------
        # Export ONNX
        # --------
        exported = False
        for opset in [18, 17]:
            try:
                with torch.no_grad():
                    torch.onnx.export(
                        model,
                        x,
                        temp_onnx,
                        opset_version=opset,
                        input_names=["input"],
                        output_names=["output"],
                        dynamo=False
                    )
                print(f"✅ ONNX exportado opset {opset}")
                exported = True
                break
            except Exception as e:
                print(f"⚠️ Opset {opset} falló:", e)

        if not exported:
            raise RuntimeError("No se pudo exportar ONNX")

        # --------
        # ONNX2TF
        # --------
        cmd = [
            sys.executable,
            "-m", "onnx2tf",
            "-i", temp_onnx,
            "-o", temp_tf_dir,
            "--non_verbose"
        ]

        result = subprocess.run(cmd, capture_output=True, text=True)

        if result.returncode != 0:
            print("⚠️ onnx2tf devolvió código distinto de 0.")
            print("STDERR:", result.stderr[:2000])

        saved_model = find_saved_model(temp_tf_dir)
        all_tflites = find_all_tflites(temp_tf_dir)

        # -------------------
        # 1) INT8 preferido
        # -------------------
        if representative_dataset is not None and saved_model:
            try:
                print("🔄 Intentando cuantización INT8...")
                converter = tf.lite.TFLiteConverter.from_saved_model(saved_model)
                converter.optimizations = [tf.lite.Optimize.DEFAULT]
                converter.representative_dataset = representative_dataset
                converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
                converter.inference_input_type = tf.int8
                converter.inference_output_type = tf.int8

                model_bytes = converter.convert()
                with open(output_path, "wb") as f:
                    f.write(model_bytes)

                if validate_tflite(output_path):
                    print("✅ INT8 generado y validado")
                    return
            except Exception as e:
                print("⚠️ INT8 falló:", e)

        # Celda 13: no copies float16 sin validar
        candidates = sorted(all_tflites, key=score_tflite, reverse=True)
        for cand in candidates:
            if validate_tflite(cand):
                shutil.copyfile(cand, output_path)
                print("✅ TFLite válido:", cand)
                return

        # Fallback float32 más limpio
        if saved_model:
            print("🔄 Fallback SavedModel -> TFLite (Float32 limpio)")
            converter = tf.lite.TFLiteConverter.from_saved_model(saved_model)
            converter.optimizations = []   # sin “optimización” rara
            model_bytes = converter.convert()
            with open(output_path, "wb") as f:
                f.write(model_bytes)

            if validate_tflite(output_path):
                print("✅ Fallback Float32 generado y validado")
                return

        raise RuntimeError("No se logró generar TFLite válido.")

    except Exception as e:
        print("❌ Error:", e)
        fallback = output_path.replace(".tflite", ".onnx")
        shutil.copyfile(temp_onnx, fallback)
        print("⚠️ Fallback ONNX guardado:", fallback)

    finally:
        if os.path.exists(temp_onnx):
            os.remove(temp_onnx)
        if os.path.exists(temp_tf_dir):
            shutil.rmtree(temp_tf_dir, ignore_errors=True)

# ------------------------
# Ejecución
# ------------------------
print("🔄 Iniciando exportación robusta...")

print("\nExportando perception...")
export_to_tflite_auto(
    student_model,
    (3, 96, 160),
    "/content/drive/MyDrive/FFAI/models/perception.tflite"
)

print("\nExportando policy...")
export_to_tflite_auto(
    policy_model,
    (32,),
    "/content/drive/MyDrive/FFAI/models/policy.tflite"
)

In [ ]:
# Celda 14: Batch Upload Receiver - Recibir experiencias del móvil

import gzip
import json
from datetime import datetime

# Storage for received experiences
received_experiences = []
BATCH_SAVE_DIR = "/content/drive/MyDrive/FFAI/batches"
os.makedirs(BATCH_SAVE_DIR, exist_ok=True)

@app.route("/upload_batch", methods=["POST"])
def upload_batch():
    """Recibe batch de experiencias comprimido desde el móvil"""
    try:
        # Recibir datos comprimidos
        compressed_data = request.data

        # Descomprimir
        json_data = gzip.decompress(compressed_data).decode('utf-8')
        batch = json.loads(json_data)

        # Guardar batch en Drive
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        batch_file = f"{BATCH_SAVE_DIR}/batch_{timestamp}.json.gz"

        with gzip.open(batch_file, 'wt') as f:
            json.dump(batch, f)

        # Agregar a memoria
        received_experiences.extend(batch)

        print(f"✅ Batch recibido: {len(batch)} experiencias")
        print(f"   Total acumulado: {len(received_experiences)} experiencias")
        print(f"   Guardado en: {batch_file}")

        return jsonify({
            "status": "success",
            "experiences_received": len(batch),
            "total_experiences": len(received_experiences)
        })

    except Exception as e:
        print(f"❌ Error recibiendo batch: {e}")
        return jsonify({"status": "error", "message": str(e)}), 500

@app.route("/stats", methods=["GET"])
def get_stats():
    """Estadísticas de entrenamiento"""
    return jsonify({
        "total_experiences": len(received_experiences),
        "batches_saved": len([f for f in os.listdir(BATCH_SAVE_DIR) if f.endswith('.json.gz')]),
        "model_versions": ["v1.0"],
        "last_training": None
    })

print("✅ Endpoints de batch upload añadidos")
print("   POST /upload_batch - Recibe experiencias del móvil")
print("   GET /stats - Estadísticas de entrenamiento")

In [ ]:
# Celda 15: Model Distribution - Descargar modelos TFLite

from flask import send_file, abort, jsonify
import os

# Nota: Esta celda debe ejecutarse ANTES de iniciar el servidor en la Celda 19
MODELS_DIR = "/content/drive/MyDrive/FFAI/models"
os.makedirs(MODELS_DIR, exist_ok=True)

# Registro de versiones de modelos
model_versions = {
    "perception": {"v1": f"{MODELS_DIR}/perception.tflite"},
    "policy": {"v1": f"{MODELS_DIR}/policy.tflite"},
    "economy": {"v1": f"{MODELS_DIR}/economy.tflite"},
}

# Note: Endpoints are now consolidated in the main API cell to avoid registration errors.
print("Endpoints de distribucion registrados exitosamente.")

In [ ]:
# Celda 16: Training Loop - Entrenar con experiencias recolectadas

def train_on_experiences(experiences, model, optimizer, epochs=10):
    """Entrenar modelo con experiencias recolectadas"""
    if not experiences:
        print("⚠️ No hay experiencias para entrenar")
        return

    print(f"🎓 Entrenando con {len(experiences)} experiencias...")

    model.train()
    losses = []

    for epoch in range(epochs):
        total_loss = 0
        count = 0

        for exp in experiences:
            # Parsear experiencia
            state_features = [
                exp.get("h", 1.0),  # health
                exp.get("a", 1.0),  # ammo
                exp.get("e", 0.0),    # enemy present
                # completa aquí hasta 32 valores reales con ceros por ahora
            ]
            # Pad with zeros to reach 32 features as expected by the model
            while len(state_features) < 32:
                state_features.append(0.0)

            state = torch.tensor(state_features, dtype=torch.float32, device=device).unsqueeze(0)

            action_idx = exp.get("a_idx", 0)
            reward = exp.get("r", 0)

            # Forward
            pred = model(state)  # shape [1, 19]
            action_logits = pred[:, :15] # First 15 for actions
            meta = pred[:, 15:]          # Last 4 for meta-parameters (x, y, duration, confidence)

            # Loss (policy gradient simplificado)
            target = torch.zeros(1, 15).to(device) # Target for action_logits
            target[0, action_idx] = reward # Assign reward to the taken action

            loss = nn.MSELoss()(action_logits, target)

            # Backward
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            count += 1

        avg_loss = total_loss / max(count, 1)
        losses.append(avg_loss)

        if epoch % 2 == 0:
            print(f"   Epoch {epoch}: loss={avg_loss:.4f}")

    print(f"✅ Entrenamiento completo. Final loss: {losses[-1]:.4f}")
    return losses

# Setup optimizer
optimizer = optim.Adam(policy_model.parameters(), lr=0.001)

# Entrenar si hay experiencias
if received_experiences:
    train_on_experiences(received_experiences, policy_model, optimizer, epochs=10)
else:
    print("💡 No hay experiencias aún. El modelo usará pesos aleatorios.")
    print("   Conecta el móvil y juega para recolectar experiencias.")

In [ ]:
from threading import Thread
import time

class ModelAutoDeployer:
    """Monitorea cambios en modelos y notifica a clientes conectados"""

    def __init__(self, socketio, check_interval=60):
        self.socketio = socketio
        self.check_interval = check_interval
        self.running = False
        self.last_modifications = {}

    def start(self):
        self.running = True
        Thread(target=self._monitor, daemon=True).start()
        print("🔄 Auto-deployer iniciado")

    def _monitor(self):
        while self.running:
            try:
                for model_name, versions in model_versions.items():
                    for version, path in versions.items():
                        if os.path.exists(path):
                            current_mtime = os.path.getmtime(path)
                            key = f"{model_name}_{version}"

                            if key in self.last_modifications:
                                if current_mtime > self.last_modifications[key]:
                                    print(f"📦 Modelo actualizado: {model_name} v{version}")
                                    self._notify_clients(model_name, version)

                            self.last_modifications[key] = current_mtime

            except Exception as e:
                print(f"❌ Error en auto-deployer: {e}")

            time.sleep(self.check_interval)

    def _notify_clients(self, model_name, version):
        """Notificar a todos los clientes conectados"""
        self.socketio.emit('model_update', {
            'model': model_name,
            'version': version,
            'url': f"https://{ngrok_domain}/models/{model_name}/{version}",
            'timestamp': time.time()
        }, namespace='/ffai')

# Iniciar auto-deployer
deployer = ModelAutoDeployer(socketio)
deployer.start()

print("✅ Auto-deployer configurado")
print("   - Monitorea /content/drive/MyDrive/FFAI/models/")
print("   - Notifica a móviles cuando hay nuevos modelos")
print("   - Intervalo de chequeo: 60 segundos")

In [ ]:
# Celda 18: Resumen y Uso del Sistema de Entrenamiento

print("""
╔════════════════════════════════════════════════════════════════╗
║         🎓 FFAI Training Pipeline - Fase 6 (Cloud)             ║
╠════════════════════════════════════════════════════════════════╣

📊 FLUJO DE TRABAJO:

1. RECOLECCIÓN (Móvil → Colab):
   - El móvil juega y recolecta experiencias
   - Batch upload cada 5 minutos o 1000 experiencias
   - POST /upload_batch recibe y guarda en Drive

2. ENTRENAMIENTO (Colab):
   - Teacher CNN: Modelo grande para detección
   - Student CNN: Destilado del teacher (~2MB)
   - Policy MLP: Decisiones tácticas (~1MB)
   - Celda 16: Ejecutar training loop

3. EXPORTACIÓN (PyTorch → TFLite):
   - Quantization INT8 para reducir tamaño 4x
   - Export a /content/drive/MyDrive/FFAI/models/
   - Celda 13: Exportar a TFLite

4. DISTRIBUCIÓN (Colab → Móvil):
   - Auto-deployer notifica a móviles
   - GET /models/latest lista URLs
   - Móvil descarga y actualiza modelos

═══════════════════════════════════════════════════════════════

🚀 COMANDOS RÁPIDOS:

# Ver estadísticas:
!curl https://TU_NGROK_URL/stats

# Listar modelos:
!curl https://TU_NGROK_URL/models/list

# Descargar modelo:
!curl -o perception.tflite https://TU_NGROK_URL/models/perception/v1

═══════════════════════════════════════════════════════════════

📈 MONITOREO:
""")

# Mostrar estado actual
print(f"   Experiencias recibidas: {len(received_experiences)}")
print(f"   Batches guardados: {len([f for f in os.listdir(BATCH_SAVE_DIR) if f.endswith('.json.gz')])}")
print(f"   Modelos disponibles: {len([f for f in os.listdir(MODELS_DIR) if f.endswith('.tflite')])}")
print(f"   URL pública: https://{ngrok_domain}")

print("""
═══════════════════════════════════════════════════════════════

💡 SIGUIENTES PASOS:

1. Conecta tu móvil y juega una partida
2. Verifica que lleguen experiencias (GET /stats)
3. Ejecuta Celda 16 para entrenar
4. Ejecuta Celda 13 para exportar a TFLite
5. El móvil descargará automáticamente los nuevos modelos

═══════════════════════════════════════════════════════════════
""")

In [50]:
print("Iniciando servidor FFAI...")
print(f"   URL Publica: https://{ngrok_domain}")
# Start server
socketio.run(app, host="0.0.0.0", port=5000, debug=False, allow_unsafe_werkzeug=True)

In [ ]:
print("Iniciando servidor FFAI...")
print(f"   URL Publica: https://{ngrok_domain}")
# Start server
socketio.run(app, host="0.0.0.0", port=5000, debug=False, allow_unsafe_werkzeug=True)

In [ ]:
print("Iniciando servidor FFAI...")
print(f"   URL Publica: https://{ngrok_domain}")
# Start server
socketio.run(app, host="0.0.0.0", port=5000, debug=False, allow_unsafe_werkzeug=True)

In [ ]:
print("Iniciando servidor FFAI...")
print(f"   URL Publica: https://{ngrok_domain}")
# Start server
socketio.run(app, host="0.0.0.0", port=5000, debug=False, allow_unsafe_werkzeug=True)

In [51]:
## Bonus: Guardar Checkpoint y Exportar Modelos

# Ejecuta esto para guardar manualmente los modelos entrenados:

# Guardar PyTorch checkpoint
import time
import os
import torch

checkpoint_path = "/content/drive/MyDrive/FFAI/checkpoints/trained_model.pth"
os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)

torch.save({
    'teacher_state': teacher_model.state_dict() if 'teacher_model' in globals() else None,
    'student_state': student_model.state_dict() if 'student_model' in globals() else None,
    'policy_state': policy_model.state_dict() if 'policy_model' in globals() else None,
    'optimizer_state': optimizer.state_dict() if 'optimizer' in globals() else None,
    'experiences_count': len(received_experiences),
    'timestamp': time.time()
}, checkpoint_path)

print(f"✅ Checkpoint guardado: {checkpoint_path}")

# Exportar a TFLite si aún no lo has hecho
if os.path.exists("/content/drive/MyDrive/FFAI/models/perception.tflite"):
    size_mb = os.path.getsize("/content/drive/MyDrive/FFAI/models/perception.tflite") / 1024 / 1024
    print(f"📦 perception.tflite: {size_mb:.2f}MB")
else:
    print("⚠️ No hay modelos TFLite exportados. Ejecuta Celda 13.")

# Verificar que los endpoints están configurados
print("\n📡 Endpoints disponibles:")
print("   POST /upload_batch - Recibir experiencias")
print("   GET  /models/list - Listar modelos")
print("   GET  /stats - Estadísticas")

✅ Checkpoint guardado: /content/drive/MyDrive/FFAI/checkpoints/trained_model.pth
📦 perception.tflite: 3.80MB

📡 Endpoints disponibles:
   POST /upload_batch - Recibir experiencias
   GET  /models/list - Listar modelos
   GET  /stats - Estadísticas
